# Sales Forecasting using MLOps Pipeline

## Business Context

A sales forecast predicts future revenue based on historical data, industry trends, and the current sales pipeline. It is an essential function for organizations as it supports planning, decision-making, and risk reduction.

Accurate forecasting enables different business functions such as sales, operations, and supply chain to align their strategies. It helps estimate future demand, optimize inventory, and improve overall operational efficiency.

---

## Objective

The objective of this project is to build an end-to-end automated MLOps pipeline for generating accurate and reliable sales forecasts.

The solution will focus on:

- Automating data ingestion and preprocessing  
- Training and evaluating machine learning models  
- Registering and deploying the best-performing model  
- Enabling continuous integration and deployment (CI/CD)  
- Ensuring scalability and reproducibility  

This will help organizations make better decisions, reduce manual effort, and improve forecasting accuracy.

---

## Dataset Description

The dataset contains information about products and stores.

### Product Features

- Product_Id: Unique identifier for each product  
- Product_Weight: Weight of the product  
- Product_Sugar_Content: Sugar level classification  
- Product_Allocated_Area: Shelf space allocated to the product  
- Product_Type: Category of the product  
- Product_MRP: Maximum retail price  

### Store Features

- Store_Id: Unique identifier for each store  
- Store_Establishment_Year: Year the store was established  
- Store_Size: Size of the store  
- Store_Location_City_Type: Tier classification of the city  
- Store_Type: Type of store  

### Target Variable

- Product_Store_Sales_Total: Total sales revenue for a product in a store  

---

## Project Approach

The project will follow a structured MLOps workflow:

1. Data registration using Hugging Face  
2. Data cleaning and preparation  
3. Model training and evaluation  
4. Model registration in the Hugging Face model hub  
5. Deployment using Streamlit and Docker  
6. Automation using GitHub Actions  

---

## Expected Outcome

At the end of this project, the following will be achieved:

- A trained machine learning model for sales prediction  
- A deployed application for generating predictions  
- An automated pipeline for continuous updates  
- A scalable and production-ready system  

---

## Tools and Technologies

- Python  
- Pandas and NumPy  
- Scikit-learn or XGBoost  
- Hugging Face  
- Streamlit  
- Docker  
- GitHub Actions  

In [ ]:
# Install required libraries
!pip install pandas numpy scikit-learn huggingface_hub datasets

In [ ]:
import pandas as pd

# Load dataset from correct path
df = pd.read_csv('/content/SuperKart.csv')

# Preview data
df.head()

,Product_Id,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Id,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Product_Store_Sales_Total
0,FD6114,12.66,Low Sugar,0.027,Frozen Foods,117.08,OUT004,2009,Medium,Tier 2,Supermarket Type2,2842.40
1,FD7839,16.54,Low Sugar,0.144,Dairy,171.43,OUT003,1999,Medium,Tier 1,Departmental Store,4830.02
2,FD5075,14.28,Regular,0.031,Canned,162.08,OUT001,1987,High,Tier 2,Supermarket Type1,4130.16
3,FD8233,12.10,Low Sugar,0.112,Baking Goods,186.31,OUT001,1987,High,Tier 2,Supermarket Type1,4132.18
4,NC1180,9.57,No Sugar,0.010,Health and Hygiene,123.67,OUT002,1998,Small,Tier 3,Food Mart,2279.36


In [ ]:
print("Shape:", df.shape)

Shape: (8763, 12)


In [ ]:
df.dtypes

,0
Product_Id,object
Product_Weight,float64
Product_Sugar_Content,object
Product_Allocated_Area,float64
Product_Type,object
Product_MRP,float64
Store_Id,object
Store_Establishment_Year,int64
Store_Size,object
Store_Location_City_Type,object


In [ ]:
df.isnull().sum()

,0
Product_Id,0
Product_Weight,0
Product_Sugar_Content,0
Product_Allocated_Area,0
Product_Type,0
Product_MRP,0
Store_Id,0
Store_Establishment_Year,0
Store_Size,0
Store_Location_City_Type,0


## Initial Data Inspection

The dataset was successfully loaded from the local environment.

Observations:

- The dataset contains both numerical and categorical variables.
- The structure includes product-level and store-level features.
- Missing values are present in some columns and will require preprocessing.
- The dataset size is adequate for building a predictive model.

These checks help in understanding the dataset before performing cleaning and modeling.

In [ ]:
import os
from huggingface_hub import login

os.environ["HF_TOKEN"] = "hf_xxxxx"

login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [28]:
df.to_csv("superkart_clean.csv", index=False)

## Data Registration

The dataset was successfully uploaded to the Hugging Face dataset hub.

The dataset was registered by creating a repository and uploading the data file directly through the Hugging Face interface.

This ensures centralized data access, version control, and reproducibility for the MLOps pipeline.

In [29]:
from datasets import load_dataset

# Load dataset from Hugging Face
dataset = load_dataset("psr1992/superkart-sales-data")

# Convert to pandas
df = dataset['train'].to_pandas()

df.head()

superkart_clean.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8763 [00:00<?, ? examples/s]

,Product_Id,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Id,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Product_Store_Sales_Total
0,FD6114,12.66,Low Sugar,0.027,Frozen Foods,117.08,OUT004,2009,Medium,Tier 2,Supermarket Type2,2842.40
1,FD7839,16.54,Low Sugar,0.144,Dairy,171.43,OUT003,1999,Medium,Tier 1,Departmental Store,4830.02
2,FD5075,14.28,Regular,0.031,Canned,162.08,OUT001,1987,High,Tier 2,Supermarket Type1,4130.16
3,FD8233,12.10,Low Sugar,0.112,Baking Goods,186.31,OUT001,1987,High,Tier 2,Supermarket Type1,4132.18
4,NC1180,9.57,No Sugar,0.010,Health and Hygiene,123.67,OUT002,1998,Small,Tier 3,Food Mart,2279.36


In [30]:
# Check missing values
df.isnull().sum()

,0
Product_Id,0
Product_Weight,0
Product_Sugar_Content,0
Product_Allocated_Area,0
Product_Type,0
Product_MRP,0
Store_Id,0
Store_Establishment_Year,0
Store_Size,0
Store_Location_City_Type,0


In [31]:
# Fill missing Product_Weight with median
df['Product_Weight'].fillna(df['Product_Weight'].median(), inplace=True)

# Replace 'Not given' in Product_Sugar_Content with mode
df['Product_Sugar_Content'] = df['Product_Sugar_Content'].replace('Not given', df['Product_Sugar_Content'].mode()[0])

/tmp/ipykernel_4840/3522468829.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Product_Weight'].fillna(df['Product_Weight'].median(), inplace=True)


In [32]:
# Create store age
df['Store_Age'] = 2026 - df['Store_Establishment_Year']

In [33]:
df.drop(['Product_Id', 'Store_Id'], axis=1, inplace=True)

In [34]:
from sklearn.model_selection import train_test_split

X = df.drop('Product_Store_Sales_Total', axis=1)
y = df['Product_Store_Sales_Total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
train_df = X_train.copy()
train_df['Product_Store_Sales_Total'] = y_train

test_df = X_test.copy()
test_df['Product_Store_Sales_Total'] = y_test

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

## Data Preparation

The dataset was loaded from the Hugging Face repository and converted into a pandas DataFrame.

Data cleaning steps performed:

- Missing values in Product_Weight were handled using median imputation  
- Invalid values in Product_Sugar_Content were replaced with the most frequent category  
- A new feature Store_Age was created from Store_Establishment_Year  
- Irrelevant identifier columns were removed  

The dataset was then split into training and testing sets using an 80:20 ratio.

The processed datasets were saved as train.csv and test.csv for further use.

## Train-Test Data Upload

The cleaned dataset was split into training and testing sets.

- Training set (80%) used for model training  
- Testing set (20%) used for model evaluation  

Both datasets were uploaded to the Hugging Face dataset repository to ensure centralized access and reproducibility in the MLOps pipeline.

In [36]:
# Import library to load dataset from Hugging Face
from datasets import load_dataset

# Load dataset from your Hugging Face repository
dataset = load_dataset("psr1992/superkart-sales-data")

# Convert train and test splits into pandas DataFrames
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

# Preview training data
train_df.head()

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/7010 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1753 [00:00<?, ? examples/s]

,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Store_Age,Product_Store_Sales_Total
0,13.64,No Sugar,0.071,Household,157.10,2009,Medium,Tier 2,Supermarket Type2,17,3882.69
1,10.03,Low Sugar,0.016,Frozen Foods,126.66,1987,High,Tier 2,Supermarket Type1,39,2444.17
2,12.26,Low Sugar,0.140,Snack Foods,138.27,2009,Medium,Tier 2,Supermarket Type2,17,3185.13
3,10.28,Low Sugar,0.143,Canned,147.44,2009,Medium,Tier 2,Supermarket Type2,17,2926.08
4,16.38,Low Sugar,0.082,Baking Goods,205.16,1999,Medium,Tier 1,Departmental Store,27,5485.40


In [37]:
# Target variable = what we want to predict
target_column = 'Product_Store_Sales_Total'

# Features = all columns except target
X_train = train_df.drop(target_column, axis=1)
y_train = train_df[target_column]

X_test = test_df.drop(target_column, axis=1)
y_test = test_df[target_column]

In [38]:
# Machine learning models cannot understand text directly
# So we convert categorical variables into numerical format using One-Hot Encoding

import pandas as pd

# Apply one-hot encoding
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Ensure both train and test have SAME columns
# This avoids errors during prediction
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [39]:
# Import Random Forest model
from sklearn.ensemble import RandomForestRegressor

# Initialize model with default parameters
model = RandomForestRegressor(random_state=42)

# Train the model using training data
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [40]:
# Use trained model to predict on test data
y_pred = model.predict(X_test)

In [41]:
# Import evaluation metrics
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# RMSE (Root Mean Squared Error) → measures prediction error
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R2 Score → measures how well model explains variance
r2 = r2_score(y_test, y_pred)

# Print results
print("Baseline Model Performance:")
print("RMSE:", rmse)
print("R2 Score:", r2)

Baseline Model Performance:
RMSE: 285.0669789309648
R2 Score: 0.9287802291704675


In [42]:
# Improve model by tuning parameters

model_tuned = RandomForestRegressor(
    n_estimators=200,   # number of trees
    max_depth=10,       # limit tree depth to avoid overfitting
    random_state=42
)

# Train tuned model
model_tuned.fit(X_train, y_train)

# Predict again
y_pred_tuned = model_tuned.predict(X_test)

# Evaluate tuned model
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\nTuned Model Performance:")
print("RMSE:", rmse_tuned)
print("R2 Score:", r2_tuned)


Tuned Model Performance:
RMSE: 282.122124964435
R2 Score: 0.9302440852740604


## Model Training and Evaluation

The training and testing datasets were loaded from the Hugging Face repository.

The following steps were performed:

- Features and target variable were separated  
- Categorical variables were converted using one-hot encoding  
- Training and testing datasets were aligned to ensure consistency  

A Random Forest Regressor was used as the baseline model.

Model performance was evaluated using RMSE and R² score.

Further improvement was achieved by tuning key hyperparameters such as the number of estimators and maximum depth.

The tuned model showed better performance compared to the baseline model.

In [43]:
import joblib

# Save the tuned model locally
joblib.dump(model_tuned, "sales_forecast_model.pkl")

['sales_forecast_model.pkl']

## Model Registration

The trained machine learning model was saved locally using joblib.

A model repository was created on Hugging Face, and the trained model was uploaded.

This allows centralized storage, version control, and easy access for deployment.

The registered model can now be used for inference in production environments.

In [46]:
with open("app.py", "w") as f:
    f.write("""
import gradio as gr
import pandas as pd
import joblib

# Load trained model
model = joblib.load("sales_forecast_model.pkl")

# IMPORTANT: define expected columns (based on training)
# We will simulate input with correct structure

def predict(product_weight, product_mrp, store_age):

    # Create dataframe with required columns
    data = {
        'Product_Weight': product_weight,
        'Product_MRP': product_mrp,
        'Store_Age': store_age
    }

    df = pd.DataFrame([data])

    # NOTE: model was trained on many encoded columns
    # So we align structure by reindexing
    model_features = model.feature_names_in_

    df = pd.get_dummies(df)

    df = df.reindex(columns=model_features, fill_value=0)

    prediction = model.predict(df)

    return float(prediction[0])


interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Product Weight"),
        gr.Number(label="Product MRP"),
        gr.Number(label="Store Age")
    ],
    outputs="number",
    title="Sales Forecast Prediction App"
)

interface.launch()
""")

In [47]:
with open("requirements.txt", "w") as f:
    f.write("""
gradio
pandas
scikit-learn
joblib
""")

## Model Deployment

The trained model was deployed using Gradio on Hugging Face Spaces.

A user interface was created to accept input features and generate predictions in real time.

The application loads the trained model, processes user inputs, and returns predicted sales output.

This enables interactive and real-time usage of the model.